# GENOME SENTINEL AI
**Complete one-cell research prototype.** Default mode uses explicitly synthetic AMR-like data. For real research, configure a curated gene-presence matrix and validated phenotype labels.

In [ ]:
# GENOME SENTINEL AI — COMPLETE ONE-CELL PROTOTYPE
# AMR gene-pattern exploration. Default data are SYNTHETIC and for software demonstration only.
# For real research, set USE_REAL_CSV=True and supply a curated 0/1 gene matrix plus validated phenotype.

import sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")
for m,p in {"numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","seaborn":"seaborn","sklearn":"scikit-learn","networkx":"networkx"}.items():
    if importlib.util.find_spec(m) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, networkx as nx
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report,confusion_matrix,roc_auc_score,
    accuracy_score,precision_score,recall_score,f1_score,RocCurveDisplay,silhouette_score)
from sklearn.inspection import permutation_importance

SEED=42
np.random.seed(SEED)
OUT=Path("genome_sentinel_outputs"); OUT.mkdir(exist_ok=True)

# ---------- CONFIG: change these values for your own curated dataset ----------
USE_REAL_CSV=False
CSV_PATH="amr_gene_matrix.csv"
ID_COL="isolate_id"
TARGET_COL="phenotype"  # binary: 1=resistant, 0=susceptible

# ---------- LOAD REAL DATA OR CREATE EXPLICITLY SYNTHETIC DEMO DATA ----------
if USE_REAL_CSV:
    raw=pd.read_csv(CSV_PATH)
    if TARGET_COL not in raw: raise ValueError(f"Missing target column: {TARGET_COL}")
    if ID_COL not in raw: raw[ID_COL]=[f"sample_{i+1}" for i in range(len(raw))]
    ids=raw[ID_COL].astype(str)
    y=pd.to_numeric(raw[TARGET_COL],errors="coerce")
    Xdf=raw.drop(columns=[ID_COL,TARGET_COL]).select_dtypes(include="number")
    valid=y.notna(); Xdf=Xdf.loc[valid]; y=y.loc[valid].astype(int); ids=ids.loc[valid]
    data_note="REAL USER-SUPPLIED DATA — verify source, labels and permissions."
else:
    rng=np.random.default_rng(SEED); n=240
    genes=["blaTEM","blaSHV","blaCTX_M","blaOXA","mecA","vanA","vanB","ermB",
           "tetA","tetM","sul1","sul2","dfrA","qnrS","aac6Ib","aph3III",
           "gyrA_mut","parC_mut","intI1","traA","repA","acrB","marA","ompF_loss"]
    A=rng.binomial(1,.34,(n,len(genes)))
    for i in range(n):
        if A[i,2] or A[i,0]: A[i,8]=rng.binomial(1,.68); A[i,10]=rng.binomial(1,.55)
        if A[i,4]: A[i,7]=rng.binomial(1,.60)
        if A[i,18]: A[i,10]=max(A[i,10],rng.binomial(1,.75)); A[i,12]=max(A[i,12],rng.binomial(1,.55))
    signal=1.2*A[:,0]+1.4*A[:,2]+1.5*A[:,4]+1.2*A[:,5]+.8*A[:,8]+.9*A[:,10]+.7*A[:,13]+.9*A[:,14]+.6*A[:,17]
    prob=1/(1+np.exp(-(signal-3)))
    y=pd.Series((rng.random(n)<prob).astype(int),name=TARGET_COL)
    Xdf=pd.DataFrame(A,columns=genes); ids=pd.Series([f"SYN_{i+1:03d}" for i in range(n)])
    data_note="SYNTHETIC DEMONSTRATION DATA — not biological evidence."

Xdf=Xdf.apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
Xdf=Xdf.loc[:,Xdf.nunique(dropna=True)>1]
if len(Xdf)<20 or Xdf.shape[1]<2: raise ValueError("Need at least 20 samples and 2 variable numeric gene features.")
if set(pd.Series(y).unique())-{0,1} or pd.Series(y).nunique()!=2: raise ValueError("Phenotype must be binary: 0/1.")
print("GENOME SENTINEL AI |",data_note,"\nSamples:",len(Xdf)," | Gene/features:",Xdf.shape[1])
print(pd.Series(y).value_counts().sort_index().rename(index={0:"Class 0 / susceptible",1:"Class 1 / resistant"}))

# ---------- EDA ----------
print("\nMissingness (%):"); display((Xdf.isna().mean()*100).sort_values(ascending=False).head(15).round(2))
print("\nDescriptive statistics:"); display(Xdf.describe().T.round(3))
fig,ax=plt.subplots(1,3,figsize=(16,4))
pd.Series(y).value_counts().sort_index().plot.bar(ax=ax[0],title="Phenotype class balance")
ax[0].set_xlabel("0=susceptible, 1=resistant"); ax[0].set_ylabel("Samples")
(Xdf.isna().mean()*100).sort_values().tail(12).plot.barh(ax=ax[1],title="Top missingness (%)")
Xdf.mean().sort_values().tail(12).plot.barh(ax=ax[2],title="Most prevalent gene features")
plt.tight_layout(); plt.show()

prevalence=Xdf.mean().sort_values(ascending=False).rename("prevalence")
assoc=pd.DataFrame({"class_0_mean":Xdf.loc[np.asarray(y)==0].mean(),
                    "class_1_mean":Xdf.loc[np.asarray(y)==1].mean()})
assoc["difference"]=assoc.class_1_mean-assoc.class_0_mean
print("Gene prevalence:"); display(prevalence.head(15).to_frame())
print("Largest descriptive class prevalence differences:"); display(assoc.sort_values("difference",ascending=False).head(15).round(3))
plt.figure(figsize=(12,9)); sns.heatmap(Xdf.fillna(Xdf.median()).corr(),cmap="vlag",center=0)
plt.title("Gene-feature correlation / co-occurrence heatmap"); plt.tight_layout(); plt.show()

# ---------- IMPUTATION, SCALING, CLUSTERING ----------
imputer=SimpleImputer(strategy="most_frequent")
Xi=pd.DataFrame(imputer.fit_transform(Xdf),columns=Xdf.columns,index=Xdf.index)
scaler=StandardScaler(); Xs=scaler.fit_transform(Xi)
ks=[]
for k in range(2,min(7,len(Xi)-1)):
    lab=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit_predict(Xs)
    ks.append({"k":k,"silhouette":silhouette_score(Xs,lab)})
ks=pd.DataFrame(ks); display(ks.round(3))
best_k=int(ks.loc[ks.silhouette.idxmax(),"k"])
km=KMeans(n_clusters=best_k,n_init=30,random_state=SEED)
clusters=km.fit_predict(Xs)
pca=PCA(n_components=2,random_state=SEED); xy=pca.fit_transform(Xs)
plt.figure(figsize=(8,6)); sc=plt.scatter(xy[:,0],xy[:,1],c=clusters,cmap="tab10",s=48,alpha=.8)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})"); plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title(f"K-Means gene-profile clusters (k={best_k})"); plt.colorbar(sc,label="Cluster"); plt.tight_layout(); plt.show()
profiles=Xi.assign(cluster=clusters).groupby("cluster").mean().T
print("Cluster gene profiles:"); display(profiles.round(2))

# ---------- GENE CO-OCCURRENCE NETWORK ----------
B=(Xi>=.5).astype(int); co=B.T@B; G=nx.Graph(); G.add_nodes_from(B.columns)
threshold=max(3,int(.05*len(B)))
for i,g1 in enumerate(B.columns):
    for g2 in B.columns[i+1:]:
        if co.loc[g1,g2]>=threshold: G.add_edge(g1,g2,weight=int(co.loc[g1,g2]))
plt.figure(figsize=(11,8)); pos=nx.spring_layout(G,seed=SEED)
weights=[d["weight"] for _,_,d in G.edges(data=True)]
nx.draw_networkx_nodes(G,pos,node_size=750,node_color="#9bd3e6")
nx.draw_networkx_edges(G,pos,width=[.5+3*w/max(weights or [1]) for w in weights],alpha=.45)
nx.draw_networkx_labels(G,pos,font_size=8); plt.title(f"Gene co-occurrence network (shared in ≥{threshold} samples)")
plt.axis("off"); plt.tight_layout(); plt.show()
network=pd.DataFrame({"gene":list(G.nodes),"degree":[G.degree(v) for v in G.nodes],
 "weighted_degree":[sum(d["weight"] for _,_,d in G.edges(v,data=True)) for v in G.nodes]}).sort_values("weighted_degree",ascending=False)
display(network.head(15))

# ---------- ANOMALY SCREEN ----------
iso=IsolationForest(n_estimators=300,contamination="auto",random_state=SEED)
flags=iso.fit_predict(Xs)==-1; anomaly_score=-iso.score_samples(Xs)
anomaly=pd.DataFrame({"sample_id":np.asarray(ids),"anomaly_flag":flags,
 "anomaly_score":anomaly_score,"cluster":clusters,"phenotype":np.asarray(y)}).sort_values("anomaly_score",ascending=False)
print("Most statistically unusual samples (not clinical risks or errors):"); display(anomaly.head(12))

# ---------- SUPERVISED BENCHMARK ----------
# For repeated isolates from patients, use patient-grouped splitting instead of random stratification.
Xtr,Xte,ytr,yte=train_test_split(Xi,pd.Series(y).reset_index(drop=True),test_size=.25,
                                random_state=SEED,stratify=pd.Series(y).reset_index(drop=True))
model=RandomForestClassifier(n_estimators=400,max_features="sqrt",min_samples_leaf=2,
                             class_weight="balanced",random_state=SEED,n_jobs=-1)
model.fit(Xtr,ytr); pred=model.predict(Xte); probas=model.predict_proba(Xte)[:,1]
print("\nClassification report:\n",classification_report(yte,pred,zero_division=0))
print("Accuracy:",round(accuracy_score(yte,pred),3)," Precision:",round(precision_score(yte,pred,zero_division=0),3),
      " Recall:",round(recall_score(yte,pred,zero_division=0),3)," F1:",round(f1_score(yte,pred,zero_division=0),3),
      " ROC-AUC:",round(roc_auc_score(yte,probas),3))
fig,ax=plt.subplots(1,2,figsize=(11,4))
sns.heatmap(confusion_matrix(yte,pred),annot=True,fmt="d",cmap="Blues",ax=ax[0]); ax[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(yte,probas,ax=ax[1]); ax[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

perm=permutation_importance(model,Xte,yte,n_repeats=15,random_state=SEED,scoring="roc_auc")
imp=pd.Series(perm.importances_mean,index=Xte.columns).sort_values().tail(15)
plt.figure(figsize=(8,6)); imp.plot.barh(); plt.title("Permutation feature importance (test set)")
plt.xlabel("Mean decrease in ROC-AUC"); plt.tight_layout(); plt.show()

# ---------- EXPORT ----------
results=pd.DataFrame({"sample_id":np.asarray(ids),"phenotype":np.asarray(y),"cluster":clusters,
                      "anomaly_flag":flags,"anomaly_score":anomaly_score})
results.to_csv(OUT/"sample_level_results.csv",index=False)
prevalence.to_csv(OUT/"gene_prevalence.csv"); assoc.to_csv(OUT/"phenotype_feature_comparison.csv")
profiles.to_csv(OUT/"cluster_profiles.csv"); network.to_csv(OUT/"gene_network_metrics.csv",index=False)
ks.to_csv(OUT/"cluster_diagnostics.csv",index=False)
pd.DataFrame({"feature":Xte.columns,"permutation_importance":perm.importances_mean}).to_csv(OUT/"feature_importance.csv",index=False)
print("\nOutputs exported to:",OUT.resolve())
print("\nRESEARCH CAUTION: Synthetic mode demonstrates code only. It is not a biological discovery or clinical prediction.")
print("For real data, document organism, accession, assay, phenotype definitions, breakpoint standard/version,")
print("sample relatedness, class balance, missingness and independent validation. Never use this prototype to guide treatment.")
